# 00 Run All Colab

Standalone, clean-kernel execution. Shared bootstrap installs and verifies dependencies, initializes all variables, and discovers ZIP/codebook inputs. NEW_PROTOCOL is explicit; historical results are preserved. Synthetic outputs are software validation only. Master executes core stages audit → cohort → descriptive → inference → split → development → report. It does not execute the other notebooks or the entire post-audit grid. Final Test is opt-in, disabled by default.

In [1]:
import os
import sys
from pathlib import Path
import importlib.util

SYNTHETIC = os.environ.get("BRFSS_SYNTHETIC", "0") == "1"
if not SYNTHETIC:
    from google.colab import drive
    drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
explicit = os.environ.get("BRFSS_REPO")
candidates = [Path(explicit).expanduser()] if explicit else [
    Path.cwd(), Path.cwd().parent,
    drive_root / "PPTK-main (1)/PPTK-main/BRFSS2015-Statistical-Methods",
    drive_root / "BRFSS2015-Statistical-Methods",
    drive_root / "BRFSS2015_Statistical_Methods_AZ/BRFSS2015-Statistical-Methods",
]
if not explicit and drive_root.is_dir():
    candidates += sorted(drive_root.glob("PPTK-main*/PPTK-main/BRFSS2015-Statistical-Methods"))
REPO = next((p.resolve() for p in candidates if (p / "pyproject.toml").is_file() and (p / "src/pipeline.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError(f"Set BRFSS_REPO to a complete project folder. Checked: {candidates}")
bootstrap_file = REPO / "scripts/notebook_bootstrap.py"
if not bootstrap_file.is_file():
    raise FileNotFoundError(f"Missing {bootstrap_file}; update the entire repository.")
spec = importlib.util.spec_from_file_location("brfss_bootstrap", bootstrap_file)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
globals().update(bootstrap.initialize(require_data=True))
print("Repository:", REPO, "Python:", sys.version.split()[0], "Protocol:", PROTOCOL)


No broken requirements found.

{
  "python": "3.13.17",
  "numpy": "2.2.6",
  "pandas": "2.2.3",
  "scipy": "1.15.3",
  "scikit-learn": "1.6.1",
  "statsmodels": "0.14.4",
  "imbalanced-learn": "0.13.0",
  "matplotlib": "3.10.1",
  "seaborn": "0.13.2",
  "PyYAML": "6.0.2",
  "pyreadstat": "1.2.8",
  "nbformat": "5.10.4",
  "nbclient": "0.10.2",
  "ipykernel": "6.29.5",
  "pytest": "8.3.5",
  "ruff": "0.11.13",
  "requests": "2.32.3",
  "pypdf": "5.5.0"
}
Repository: /workspace/BRFSS2015-Statistical-Methods Python: 3.13.17 Protocol: NEW_PROTOCOL


In [2]:
from scripts.colab_runtime import execute
from src.reporting.report import generate
from IPython.display import display, Markdown
if PROTOCOL != "NEW_PROTOCOL":
    from scripts.colab_runtime import require_legacy
    require_legacy()
    display(Markdown((REPO / "reports/FINAL_RESEARCH_REPORT.md").read_text()))
else:
    through = "report" if ALLOW_FINAL and True else "develop"
    execute(protocol=PROTOCOL, synthetic=SYNTHETIC, data=DATA, codebook=CODEBOOK_PATH,
            out=OUT, profile=PROFILE, mode=MODE, seed=SEED, through=through, allow_final=ALLOW_FINAL and True)
    generate(OUT, synthetic=SYNTHETIC)
    display(Markdown((OUT / "final_report.md").read_text()))
    print("Final Test:", "previously completed; report only" if (OUT / "test_access.json").exists() else "NOT_ACCESSED")
print("Outputs:", OUT)
print("SYNTHETIC SOFTWARE VALIDATION ONLY" if SYNTHETIC else "Review current-run audit and protocol scope.")
print("Actual hosted runtime: COLAB_RUNTIME_NOT_VERIFIED by repository tests.")

2026-11-06 16:38:54,971 INFO RESUME audit


2026-11-06 16:38:54,972 INFO RESUME cohort


2026-11-06 16:38:54,973 INFO RESUME descriptive


2026-11-06 16:38:54,973 INFO RESUME inference


2026-11-06 16:38:54,978 INFO RESUME split


2026-11-06 16:38:54,984 INFO RESUME develop


# Synthetic validation report — not epidemiological evidence

## Abstract

This cross-sectional analysis investigates self-reported diabetes associations and fixed-protocol predictive classification. Estimates describe associations, not causal effects or future incident disease.

## Research questions and prespecified analysis

RQ1: adjusted associations using complex survey design. RQ2: training-only feature, imbalance and model comparisons. RQ3: balanced accuracy, sensitivity, calibration and discrimination versus a same-cohort logistic baseline. RQ4: subgroup and outcome/missingness sensitivity. Full estimands and hypotheses are in docs/RESEARCH_PROTOCOL.md.

## Data and cohort

Source records: 960. Eligible: 871. Complete cases: 662. Design strata: 120. Profile: quick. Source fingerprints and exact schema are in audit.json.

## Methods

CDC core DIABETE3=1 versus 3 is the primary contrast; pregnancy-only and borderline responses are excluded. Optional prediabetes ascertainment is not assumed complete. Survey point estimates use _LLCPWT and variance uses stratified _PSU score totals within _STSTR, without a finite population correction. Complete-case inference and median/mode sensitivity target different missing-data assumptions. Preprocessing and feature selection are fitted inside training folds. The final test set is accessed in one frozen-model batch.

## Weighted prevalence

| class | estimate | lower | upper | n |
| --- | --- | --- | --- | --- |
| 0.0 | 0.86023 | 0.8336 | 0.8832 | 871 |
| 1.0 | 0.13977 | 0.1168 | 0.1664 | 871 |

Weighted diabetes prevalence within this defined analysis domain was 13.98% (95% design CI 11.68% to 16.64%). This domain excludes unknown outcomes and may exclude borderline/pregnancy-only responses; it is not an unconditional whole-population estimate.

## Statistical associations

Adjusted OR, confidence limits, design-based p-values, and exploratory BH corrections: adjusted_or.csv. Multinomial RRR: multinomial_rrr.csv when completed. Univariate results: univariate.csv. Clinical interpretation must consider coding references, precision, multiplicity, selection, and reverse causation.

Prespecified illustrative contrasts are below; the complete coefficient table accompanies this report. OR/RRR compare coded levels with the first codebook category, holding included covariates fixed. BMI is per 10 kg/m²; no coefficient is a causal effect.

| term | OR | lower | upper | p |
| --- | --- | --- | --- | --- |
| _AGEG5YR_13 | 2.21134 | 0.6492 | 7.53241 | 0.20357 |
| SEX_2 | 1.0418 | 0.61261 | 1.77168 | 0.87949 |
| _BMI5_per_10 | 2.03451 | 1.16385 | 3.5565 | 0.01286 |
| INCOME2_8 | 0.68377 | 0.22568 | 2.07169 | 0.50031 |
| BPHIGH4_3 | 1.98819 | 0.95013 | 4.16036 | 0.06801 |
| TOLDHI2_2 | 1.25246 | 0.74569 | 2.10363 | 0.39364 |
| GENHLTH_5 | 0.63224 | 0.25793 | 1.54973 | 0.31507 |

## Model development

Development scores guided selection and threshold tuning; they are optimistic for the selected strategy. They are not final test performance. All candidates and skipped dependencies are in model_comparison.csv.

| model | selector | k | imbalance | cv_mean | threshold |
| --- | --- | --- | --- | --- | --- |
| extra_trees | original | 10 | none | 0.52483 | 0.2 |
| decision_tree | original | 10 | none | 0.5247 | 0.2 |
| logistic | original | 10 | smote | 0.52291 | 0.8 |
| logistic | tree | 10 | none | 0.52066 | 0.2 |
| xgboost | original | 10 | none | 0.51929 | 0.2 |
| random_forest | original | 10 | none | 0.5176 | 0.2 |
| logistic | original | 10 | random_under | 0.51749 | 0.5 |
| feature_ensemble | original | 10 | none | 0.51555 | 0.3 |
| logistic | original | 10 | class_weight | 0.51067 | 0.4 |
| gradient_boosting | original | 10 | none | 0.51043 | 0.5 |
| lightgbm | original | 10 | none | 0.5099 | 0.3 |
| soft_voting | original | 10 | none | 0.50392 | 0.5 |
| logistic | l1 | 10 | none | 0.50307 | 0.4 |
| logistic | mutual_info | 10 | none | 0.5029 | 0.3 |
| stacking | original | 10 | none | 0.50238 | 0.2 |

## Predictive results

[UNAVAILABLE: final evaluation has not executed; no result inferred.]

## Research gaps and interpretation

The gap matrix distinguishes inspected evidence from hypotheses. Chang et al. (2022), DOI 10.1016/j.health.2022.100118, reported 82.26% accuracy; our cohort/outcome and evaluation may differ. A higher point estimate here does not establish improvement over that publication. The implemented comparison estimates improvement over a prespecified logistic model on identical test respondents. Feature importance measures predictive utility, not causal contribution. Prediction, methodological validity, interpretability, and epidemiological evidence are separate contributions.

## Sensitivity and generalizability

See sensitivity_or.csv, borderline_sensitivity_or.csv, multinomial_rrr.csv, and subgroups.csv. Missing outputs are unavailable rather than negative findings. Core-question borderline status is an imperfect three-category self-report; module analyses require geographic/version eligibility and appropriate weights. External or temporal validation has not been performed.

## Limitations

Self-reported cross-sectional data cannot distinguish all diabetes types, verify biological disease, or determine temporal direction. Complete-case inference may be selected. Deterministic imputation sensitivity is not multiple imputation. Calibrated weights are treated as fixed in the with-replacement linearization. Predictive bootstrap intervals are conditional respondent intervals, not full survey population intervals or uncertainty over model selection. Small demographic groups may be unstable. Quick mode caps training size and is a computational validation rather than a full experimental study.

## Reproducibility

Source hashes, code/config signature, package versions, split IDs/checksum, CV indices, experiment history, frozen model checksum, test-access ledger and atomic checkpoints accompany this report. See execution_status.json; actual Google Colab execution remains unverified.

## Conclusion

All numerical results in this report are synthetic software validation. They provide no evidence about diabetes prevalence, real-world associations, or predictive benefit.

## References

CDC BRFSS 2015 data and codebook: https://www.cdc.gov/brfss/annual_data/annual_2015.html

Chang V, Ganatra MA, Hall K, Golightly L, Xu Q (2022). An assessment of machine learning models and algorithms for early prediction and diagnosis of diabetes using health indicators. Healthcare Analytics 2:100118. https://doi.org/10.1016/j.health.2022.100118


Final Test: NOT_ACCESSED
Outputs: /tmp/brfss-notebook-1d2z9bdi
SYNTHETIC SOFTWARE VALIDATION ONLY
Actual hosted runtime: COLAB_RUNTIME_NOT_VERIFIED by repository tests.
